In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pylab as plt

import seaborn as sns

from skspatial.objects import Line, Plane
from skspatial.plotting import plot_3d


from skspatial.objects import Line, Cylinder, Point, Points
from skspatial.plotting import plot_3d

import phasespace

import tensorflow

import bisect
import numpy as np
import matplotlib.pylab as plt
import pandas as pd

import seaborn as sns

import numpy as np
from sklearn.mixture import GaussianMixture
from scipy.stats import multivariate_normal

import numpy as np
from scipy.interpolate import griddata
from scipy.integrate import quad, trapezoid
from scipy.interpolate import CubicSpline

import matplotlib.pylab as plt
from scipy import stats
from matplotlib import cm
from matplotlib.ticker import LinearLocator

from scipy.interpolate import LinearNDInterpolator

import eloss_tools


import dm_generation_tools as dgt
import detector_simulation_tools as dst

import glob

import time

####################################
import warnings
# Suppress all warnings
warnings.filterwarnings("ignore")


import pickle

In [ ]:
print(phasespace.__version__)

print(tensorflow.__version__)

In [ ]:
def generate_many_events(MASSES_A=[1], MASSES_DM=[100], nevents_to_generate=10, \
                             radius=20, depth=-7.5, dm_model='floating', 
                             SAVE_ONLY_DETECTED=True, eloss_dict_name=None, 
                             save_to_file=False, additional_tag=""):

    #######################################################################
    # Generate the decays of the dark photons
    #######################################################################
    
    start_time = time.time()
    df_decays = dgt.generate_dm_decays(MASSES_A=MASSES_A, MASSES_DM=MASSES_DM, nevents_to_generate=nevents)
    print(f"\nTime to generate events {time.time() - start_time:.2} seconds  ------")

    print(f'Shape of dataframe is {df_decays.shape}')

    #######################################################################
    # We generate in x,y,z so add fields for pmag, costh, and phi
    #######################################################################

    ################### MAYBE REMOVE PT HERE SINCE THIS IS NOT REALLY WHAT WE WANT
    # Calculate pT for muon 1
    pmag = df_decays['pmag1']
    theta_rad = df_decays['theta1']
    costheta = np.cos(theta_rad)
    #print(pmag)
    
    pt = pmag*costheta
    df_decays['pt1'] = pt
    
    # Calculate pT for muon 2
    pmag = df_decays['pmag2']
    theta_rad = df_decays['theta2']
    costheta = np.cos(theta_rad)
    #print(pmag)
    
    pt = pmag*costheta
    df_decays['pt2'] = pt

    #######################################################################
    # Place the decay points
    #######################################################################

    nentries = len(df_decays)

    origins, directions = dst.generate_origins_and_directions(nevents=nentries, radius=radius, depth=depth, dm_model=dm_model)
    #print(origins)
    
    #######################################################################
    # What strikes the detector
    #######################################################################

    pts0,pts1 = dst.intersect_finite_cylinder_x_np(origins=origins, directions=directions)
    #print(pts0)
    
    hit_detector_idx = ~np.isnan(pts0.T[0])
    
    df_decays['hit_detector'] = hit_detector_idx
    
    df_decays['x0'] = origins.T[0]
    df_decays['y0'] = origins.T[1]
    df_decays['z0'] = origins.T[2]
    
    distance_to_detector = np.sqrt(df_decays['x0']**2 + df_decays['y0']**2 + df_decays['z0']**2)
    
    df_decays['distance_to_detector'] = distance_to_detector
    
    df_decays['px0'] = directions.T[0]
    df_decays['py0'] = directions.T[1]
    df_decays['pz0'] = directions.T[2]

    # Save the intersection points
    df_decays['ip_x0'] = pts0.T[0]
    df_decays['ip_y0'] = pts0.T[1]
    df_decays['ip_z0'] = pts0.T[2]

    df_decays['ip_x1'] = pts1.T[0]
    df_decays['ip_y1'] = pts1.T[1]
    df_decays['ip_z1'] = pts1.T[2]

    # Did it
    #px,py,pz = directions.T
    #pmag = np.sqrt(px**2 + py**2 + pz**2)
    #
    #theta = np.arccos(pz/pmag)
    #phi = np.arctan2(py, px)

    #######################################################################
    # Do we save only the ones that strike the detector?
    #######################################################################
    detected_tag = ""
    if SAVE_ONLY_DETECTED:
        print("Saving only the tracks that strike the detector")
        print(f"Initially dataframe is {df_decays.shape}")
        filter = df_decays['hit_detector'] == True
        df_temp = df_decays[filter]
        del df_decays
        df_decays = df_temp
        print(f"After only keeping some tracks, dataframe is {df_decays.shape}")
        detected_tag = "_HIT_DETECTOR"
        
    #######################################################################
    # Calculate eloss
    #######################################################################

    # New stuff with precomputed splines
    start = time.time()
    
    # Load the spline object from the file
    with open(eloss_dict_name, 'rb') as file_handle:
        loaded_eloss_dict = pickle.load(file_handle)
    
    print(f'Time to read energy loss spline file is {time.time() - start} seconds')

    lookup = eloss_tools.RaggedSplineIndexLookup(loaded_eloss_dict)

    ########### NEED TO DO THIS FOR BOTH MUONS!!!!!!!!! ######################
    #E_query = 10000 + 2000*np.random.random(nvals)
    #d_query = 1000*np.random.random(nvals)
    E_query = df_decays['e_mu1']
    d_query = df_decays['distance_to_detector']

    #print(d_query)
    
    E_idx, D_idx = lookup.get_indices(E_query, d_query)

    print(f'Processing eloss for {len(E_query)} muons')
    start = time.time()
    
    uniq, counts = eloss_tools.count_pairs(E_idx, D_idx)    
    print(len(uniq), len(counts), sum(counts))
    
    eloss_data = {}
    for i,(u,c) in enumerate(zip(uniq, counts)):
        #print(u,c)
        if i%1000==0:
            print(i)
        ei,di = u
        #di = u[1]
        E_index = lookup.energy_keys[ei]
        d_index = lookup.dist_keys[ei][di]
        #print(u,c,E_index, d_index)
        spl = loaded_eloss_dict[E_index][d_index]
        de_vals = eloss_tools.generate_data_from_spline(spl, c)
        #print(de_vals)
        #eloss_data[(E_index,d_index)] = de_vals
        eloss_data[(ei,di)] = de_vals
    
    print(f"Calculated all the uniq and counts")
    
    e_final_vals = []
    de = -1
    for i in range(len(E_query)):
        ei = E_idx[i]
        di = D_idx[i]
        # CHECK THIS BUT I THINK di is -1 when the distance is greater than the 
        # the max distance for that energy
        de_vals = eloss_data[(ei,di)]#.pop()
        #print(ei,di,de_vals)
        if len(de_vals)>0 and di>-1:
            de = de_vals.pop()
        else:
            # Make the delta E the same as the initial energy
            de = E_query.iloc[i]
        #print(i, len(E_query))
        efin = E_query.iloc[i] - de
        #print(E_query.iloc[i], di, de, efin)
        if efin<0:
            efin = 0
        e_final_vals.append(efin)
    
    print(f'Time to calculate eloss for {len(E_query)} muons is {time.time() - start} seconds')

    df_decays['efinal_mu1'] = e_final_vals

    #######################################################################
    # Calculate pt as seen by the detector
    #######################################################################
    #pt = np.cos(theta) * df_decays['pmag1']
    # Get the intersection points back from the dataframe after reductions
    pts0 = np.array([df_decays['ip_x0'], df_decays['ip_y0'], df_decays['ip_z0']]).T 
    pts1 = np.array([df_decays['ip_x1'], df_decays['ip_y1'], df_decays['ip_z1']]).T 

    projection,vmag = dst.projection_length_on_plane_from_points(pts0, pts1)    
    #projection,vmag = dst.projection_length_on_plane_from_points(pts0[hit_detector_idx], pts1[hit_detector_idx])

    #print(projection)
    #print(vmag)
    ######## SHOULD BE CONSISTENT ABOUT USING PMAG OR ENERGY
    transverse_scaling = projection/vmag
    pt = transverse_scaling * df_decays['pmag1']
    pt_eloss = transverse_scaling * df_decays['efinal_mu1']
    
    print(len(pt))
    print(len(pt_eloss))
    
    df_decays['pt1_detector_acceptance'] = pt
    df_decays['pt1_detector_acceptance_eloss'] = pt_eloss
    
    #df_decays['pt1_scaling_detector_acceptance'] = transverse_scaling
    
    #df_decays['costh1_detector_acceptance'] = np.cos(theta)
    #df_decays['phi1_detector_acceptance'] = phi
    #df_decays['pmag1_detector_acceptance'] = pmag

    ##########################################################################
    # Generate a tag that represents all this information
    depth_tag = dst.return_tag(depth)
    m_dm_tag = dst.return_tag(MASSES_DM)
    m_a_tag = dst.return_tag(MASSES_A)
    
    tag = f'd_{depth_tag}_r_{radius}_mDM_{m_dm_tag}_mA_{m_a_tag}_dm_model_{dm_model}{detected_tag}{additional_tag}'

    # Do we write it all out to a file?
    if save_to_file:
        outfile = f'generated_data_{tag}.parquet'
        df_decays.to_parquet(outfile)#, key='df')
        print(f'Saved to file {outfile}')
    ##########################################################################


    return df_decays, tag




In [ ]:
######################################################################

MASSES_A = [.220]
#MASSES_DM = [1000] # For comparison
MASSES_DM = [200, 1000, 1600, 2000, 3000, 6000, 7000, 10000] # For comparison
#MASSES_DM = [1600, 2000, 6000, 10000] # For comparison

#MASSES_DM = [10000] # For comparison
#MASSES_DM = [200] # For comparison

nevents = 1_000_000
# For when it is a large volume and floating, this is the largest number I can do on the 32 GB machine
# but just for 1 value of M_DM and M_A
#nevents = 10_000_000 

radius = 4000 # meters
#depth = 1000 # meters, origin of detector is 0 and the radius is 7.5
depth = [-7.5, 4000] # meters, origin of detector is 0 and the radius is 7.5

dm_model = 'floating'
#dm_model = 'core'

save_to_file = True

# Generate many
for i in range(0,100):
    print(f'{i} --------------------------------------------------------')
    df_decays,tag = generate_many_events(MASSES_A=MASSES_A, MASSES_DM=MASSES_DM, nevents_to_generate=nevents, \
                                 radius=radius, depth=depth, dm_model=dm_model, 
                                 SAVE_ONLY_DETECTED=True, eloss_dict_name='eloss_dictionary_11032025_v2.pkl', 
                                    save_to_file=save_to_file, additional_tag=f'_{i:02d}')



df_decays


In [ ]:
df_decays['efinal_mu1']

In [ ]:
df_decays['distance_to_detector']

In [ ]:
partial_tag = tag[0:-11]

print(partial_tag)



In [ ]:
partial_tag = tag[0:-3]
print(partial_tag)

files = glob.glob(f'generated_data_{partial_tag}*.parquet')

# Merge them

dfs = []
for i,filename in enumerate(files):
    print(filename)
    dfs.append( pd.read_parquet(filename))

df_decays = pd.concat(dfs)

df_decays.to_parquet(f'generated_data_{partial_tag}_COMBINED.parquet')

df_decays



In [ ]:
df_decays.iloc[0]

In [ ]:
df_decays['M_DM']

In [ ]:
filter = (df_decays['efinal_mu1']>10)
filter = filter & (df_decays['M_DM']==10000)

plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
df_decays[filter]['z0'].hist(bins=100, range=(0,8000))

plt.subplot(1,3,2)
df_decays[filter]['efinal_mu1'].hist(bins=100, range=(-100,8000))

plt.subplot(1,3,3)
df_decays[filter].plot.scatter(y='efinal_mu1', x='z0', s=0.1, ax=plt.gca())

In [ ]:
# For more direct comparison

bins=25
binrange=(0,7000)

DMstr = 'DM'
lbracket = '{'
rbracket = '}'

plt.figure(figsize=(10,6))

for mass in df_decays['M_DM'].unique():
#for mass in [200, 1000, 2000, 7000]:
#for mass in [2000, 3000, 6000, 10000]:
#for mass in [1600, 3000, 6000, 10000]:
#for mass in [3000, 10000]:

#for mass in MASSES_DM:

    filter = (df_decays['M_DM'] == mass) & (df_decays['M_A']==0.22)
    filter = filter & (df_decays['efinal_mu1'] > 10)

    filter_p = (df_decays['pmag1'] > (mass * 0.95/2)) & (df_decays['pmag1'] < (mass * 1.05/2))
    
    #plt.hist(df_decays['pt1_detector_acceptance'][filter], bins=bins, range=binrange, density=True, histtype='step', linewidth=3, label=f'$M_{lbracket}{DMstr}{rbracket}={int(mass)}$ GeV/c$^2$')
    #plt.hist(df_decays['pt1_detector_acceptance_eloss'][filter], bins=bins, range=binrange, density=True, histtype='step', linewidth=3, label=f'$M_{lbracket}{DMstr}{rbracket}={int(mass)}$ GeV/c$^2$')
    plt.hist(df_decays['pt1_detector_acceptance_eloss'][filter & filter_p], bins=bins, range=binrange, density=True, histtype='step', linewidth=3, label=f'$M_{lbracket}{DMstr}{rbracket}={int(mass)}$ GeV/c$^2$')

    #plt.hist(df_decays['pmag1'][filter], bins=bins, range=binrange, density=False, histtype='step', linewidth=3, label=f'$M_{lbracket}{DMstr}{rbracket}={int(mass)}$ GeV/c$^2$')

plt.yscale('log')
#plt.ylim(100)
plt.ylim(1e-5, 1e-2)
#plt.xlim(0,7000)
plt.xlabel(r'$\mu$ $p_T$ (GeV/c)', fontsize=18) 
plt.legend()

plotfilename = f'pt_at_detector_{tag}_P_CONSTRAINED.png'
#plotfilename = f'pt_at_detector_{tag}.png'


print(f"Saving {plotfilename}")
plt.savefig(plotfilename)

In [ ]:
mass = 10000
filter = (df_decays['M_DM'] == mass) & (df_decays['M_A']==0.22)
filter_p = (df_decays['pmag1'] > (mass * 0.95/2)) & (df_decays['pmag1'] < (mass * 1.05/2))


x1 = df_decays[filter]['pmag1']
x2 = df_decays[filter & filter_p]['pmag1']

n =    len(filter_p[filter])
ncut = len(filter_p[filter & filter_p])

print(n,ncut)#,ncut/n)

plt.hist(x1,bins=100, range=(1000,10000))
plt.hist(x2,bins=100, range=(1000,10000))

;

In [ ]:
df_decays.columns

In [ ]:
def diagnostics_plots(df_decays, tag):
    # Theta ########################################
    g = sns.displot(
    df_decays, x="theta1", row="M_DM", col="M_A",
    bins=100, binrange=(0,0.5), height=3, facet_kws=dict(margin_titles=True),
)

    g.set_titles(col_template=r"Mass A ={col_name} GeV/c$^2$", row_template=r"Mass DM ={row_name} GeV/c$^2$")
    g.set_axis_labels(r"$\theta_\mu$ degrees", fontsize=18)
    g.set(yscale='log')

    # Phi ##########################################
    g = sns.displot(
    df_decays, x="phi1", row="M_DM", col="M_A",
    bins=100, binrange=(-0.005, 0.005), height=3, aspect=1.5, facet_kws=dict(margin_titles=True),
)

    g.set_titles(col_template=r"Mass A ={col_name} GeV/c$^2$", row_template=r"Mass DM ={row_name} GeV/c$^2$")
    g.set_axis_labels(r"$\phi_\mu$ (radians)", fontsize=18)
    g.set(yscale='log')

    # oPening angl ##########################################
    g = sns.displot(
        df_decays, x="opening angle", row="M_DM", col="M_A",
        bins=100, binrange=(0, 0.5), height=3, aspect=1.5, facet_kws=dict(margin_titles=True),
    )
    
    g.set_titles(col_template=r"Mass A ={col_name} GeV/c$^2$", row_template=r"Mass DM ={row_name} GeV/c$^2$")
    g.set_axis_labels(r"Opening angle $\theta$ (degrees)", fontsize=18)
    g.set(yscale='log')


    ########################################################################
    g = sns.displot(
        df_decays, x="pt1_detector_acceptance_eloss", row="M_DM", col="M_A",
        bins=50, height=3, aspect=1.0, common_bins=True,
        facet_kws={"margin_titles":True, "sharex":True}#, sharey=True)
    )
    
    g.set_titles(col_template=r"Mass A ={col_name} GeV/c$^2$", row_template=r"Mass DM ={row_name} GeV/c$^2$")
    g.set_axis_labels(r"$p_T$ GeV/c", fontsize=18)
    g.set(yscale='log')
    
    #plt.savefig('pt_at_0_depth_no_common_binning.png')
    plt.savefig(f'pt_{tag}.png')


filter = (df_decays['M_DM']==10000)
#filter = (df_decays['M_DM']>1)

diagnostics_plots(df_decays[filter], tag=tag)

In [ ]:
def plot_decay_locations(df_decays, tag):

    for mass in df_decays['M_DM'].unique():
        
        filter = (df_decays['M_DM']==mass)
        filter = filter & (df_decays['efinal_mu1']>10)
        
        fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    
        df_decays[filter].plot.scatter(x='x0', y='y0', s=0.1, ax=axes[0], alpha=0.7)
        df_decays[filter].plot.scatter(x='x0', y='z0', s=0.1, ax=axes[1], alpha=0.7)
        df_decays[filter].plot.scatter(x='y0', y='z0', s=0.1, ax=axes[2], alpha=0.7)

        plt.title(f'M_DM = {mass} GeV/c$^2$')
        
        plt.tight_layout()


plot_decay_locations(df_decays, tag)

In [ ]:
filter = df_decays['M_DM']==200

df = df_decays[filter]
df


In [ ]:
df['pt1_detector_acceptance_eloss'].hist(bins=100)

In [ ]:
df.columns

In [ ]:
#df['z0'].hist(bins=100)
df['z0'].hist(bins=100, range=(0,10000))
#plt.yscale('log')

In [ ]:
len(df)

In [ ]:
# Question about angles. 
mass = 10000
filter = (df_decays['M_DM'] == mass) & (df_decays['M_A']==0.22)
filter_p = (df_decays['pmag1'] > (mass * 0.95/2)) & (df_decays['pmag1'] < (mass * 1.05/2))

#x1 = df_decays[filter]['theta1']
#x2 = df_decays[filter & filter_p]['theta1']

x = df_decays[filter]['px0']
y = df_decays[filter]['py0']
z = df_decays[filter]['pz0']

theta = np.arccos(z/np.sqrt(x**2 + y**2 + z**2))

n =    len(filter_p[filter])
ncut = len(filter_p[filter & filter_p])

print(n,ncut)#,ncut/n)

plt.hist(theta,bins=100, range=(-1,4))
#plt.hist(x2,bins=100, range=(-8,8))

plt.xlabel(r'p$_{\theta}$',fontsize=18)

print(partial_tag)

outfile = f'theta_M_DM_{mass}_{partial_tag}.png'
plt.savefig(outfile)


In [ ]:
df_decays = pd.read_parquet('generated_data_d_-7.5-4000_r_4000_mDM_200-10000_mA_0.22_dm_model_floating_HIT_DETECTOR_COMBINED.parquet')

In [ ]:
filter = (np.abs(df_decays['x0'])>100)
df_decays[filter]['pz_mu1'] - df_decays[filter]['pmag1']

In [ ]:
import numpy as np
from typing import Tuple

def rotate_A_so_z_aligns_with_B(
    Ax: np.ndarray, Ay: np.ndarray, Az: np.ndarray,
    Bx: np.ndarray, By: np.ndarray, Bz: np.ndarray,
    *, eps: float = 1e-15
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    For each i, rotate the vector A[i] by the unique rotation R_i that maps
    the global z-axis (0,0,1) to the direction of B[i]. Returns the rotated A.

    - If B[i] == 0, no direction is defined; by convention we return A[i] unchanged.
    - If B[i] is parallel to +z, R_i = identity.
    - If B[i] is parallel to -z, we use a 180° rotation about the x-axis (diag(1,-1,-1)).

    All operations are vectorized for large n (10^6–10^7 feasible with enough RAM).
    """
    Ax = np.asarray(Ax, dtype=float); Ay = np.asarray(Ay, dtype=float); Az = np.asarray(Az, dtype=float)
    Bx = np.asarray(Bx, dtype=float); By = np.asarray(By, dtype=float); Bz = np.asarray(Bz, dtype=float)
    if not (Ax.shape == Ay.shape == Az.shape == Bx.shape == By.shape == Bz.shape):
        raise ValueError("All inputs must have identical shapes (1D arrays of length n).")
    if Ax.ndim != 1:
        raise ValueError("Inputs must be 1D arrays.")

    # Normalize B -> v (target direction). Handle zeros with mask.
    Bnorm = np.sqrt(Bx*Bx + By*By + Bz*Bz)
    has_dir = Bnorm > eps

    vx = np.zeros_like(Bx); vy = np.zeros_like(By); vz = np.zeros_like(Bz)
    vx[has_dir] = Bx[has_dir] / Bnorm[has_dir]
    vy[has_dir] = By[has_dir] / Bnorm[has_dir]
    vz[has_dir] = Bz[has_dir] / Bnorm[has_dir]

    # cos(theta) and sin(theta) where k=(0,0,1) -> v
    c = vz                                # cos θ = k·v
    # clip for numerical safety
    c = np.clip(c, -1.0, 1.0)
    s = np.sqrt(np.maximum(0.0, 1.0 - c*c))  # |k × v|

    # Axis ω = (k × v) / |k × v| = (-vy, vx, 0)/s  (undefined if s=0)
    wx = np.zeros_like(vx); wy = np.zeros_like(vy); wz = np.zeros_like(vz)
    mask_general = s > eps
    wx[mask_general] = -vy[mask_general] / s[mask_general]
    wy[mask_general] =  vx[mask_general] / s[mask_general]
    # wz = 0 always for this specific k→v rotation

    # Rodrigues: R a = a*c + (ω × a)*s + ω (ω·a) (1-c)
    # Start with identity result
    Apx, Apy, Apz = Ax.copy(), Ay.copy(), Az.copy()

    # General case (0 < s): apply Rodrigues
    if np.any(mask_general):
        mg = mask_general
        # ω × a
        cx = wy[mg]*Az[mg] - 0.0*Ay[mg]     # wz=0
        cy = 0.0*Ax[mg] - wx[mg]*Az[mg]
        cz = wx[mg]*Ay[mg] - wy[mg]*Ax[mg]
        # ω·a
        wdot = wx[mg]*Ax[mg] + wy[mg]*Ay[mg]  # + wz*Az, but wz=0

        one_minus_c = (1.0 - c[mg])

        Apx[mg] = Ax[mg]*c[mg] + cx*s[mg] + wx[mg]*wdot*one_minus_c
        Apy[mg] = Ay[mg]*c[mg] + cy*s[mg] + wy[mg]*wdot*one_minus_c
        Apz[mg] = Az[mg]*c[mg] + cz*s[mg] + 0.0*wdot*one_minus_c  # wz=0

    # Parallel to +z: c≈+1, s≈0 -> identity (already true)

    # Parallel to -z: c≈-1, s≈0 -> 180° about x-axis: (x, y, z) -> (x, -y, -z)
    mask_anti = (np.abs(s) <= eps) & (c < 0.0) & has_dir
    if np.any(mask_anti):
        Apx[mask_anti] =  Ax[mask_anti]
        Apy[mask_anti] = -Ay[mask_anti]
        Apz[mask_anti] = -Az[mask_anti]

    # No direction (B=0): leave A unchanged (already true)
    # (Optionally, you could return a mask to report which entries had B=0.)

    return Apx, Apy, Apz


In [ ]:
import numpy as np
from numpy.testing import assert_allclose

#from your_module import rotate_A_so_z_aligns_with_B

def test_identity_when_B_is_z():
    # If B already points along +z, A should be unchanged.
    Ax = np.array([1., 0., 0.]); Ay = np.array([0., 1., 0.]); Az = np.array([0., 0., 1.])
    n = Ax.size
    Bx = np.zeros(n); By = np.zeros(n); Bz = np.ones(n)
    X, Y, Z = rotate_A_so_z_aligns_with_B(Ax, Ay, Az, Bx, By, Bz)
    assert_allclose(X, Ax); assert_allclose(Y, Ay); assert_allclose(Z, Az)

def test_map_z_to_x_direction():
    # B along +x means rotate frame so z->x; test with A=z-hat -> should become x-hat
    Ax = np.array([0.]); Ay = np.array([0.]); Az = np.array([1.])
    Bx = np.array([1.]); By = np.array([0.]); Bz = np.array([0.])
    X, Y, Z = rotate_A_so_z_aligns_with_B(Ax, Ay, Az, Bx, By, Bz)
    assert_allclose([X[0], Y[0], Z[0]], [1., 0., 0.], atol=1e-12)

def test_general_random_consistency():
    rng = np.random.default_rng(0)
    n = 1000
    Ax = rng.normal(size=n); Ay = rng.normal(size=n); Az = rng.normal(size=n)
    # Random B, avoid zeros
    Bx = rng.normal(size=n); By = rng.normal(size=n); Bz = rng.normal(size=n) + 1e-6

    # Check that rotating the z-hat yields the B direction (unit)
    Xz, Yz, Zz = rotate_A_so_z_aligns_with_B(
        np.zeros(n), np.zeros(n), np.ones(n), Bx, By, Bz
    )
    # Xz,Yz,Zz should be unit vectors aligned with B
    Bnorm = np.sqrt(Bx*Bx + By*By + Bz*Bz)
    assert_allclose(np.sqrt(Xz*Xz + Yz*Yz + Zz*Zz), 1.0, rtol=1e-12, atol=1e-12)
    # Direction check via cosine similarity
    cos = (Xz*Bx + Yz*By + Zz*Bz) / Bnorm
    assert_allclose(cos, 1.0, atol=1e-10)

def test_antiparallel_case():
    # B = -z means 180° about x-axis -> (x,y,z)->(x,-y,-z)
    Ax = np.array([2.0]); Ay = np.array([3.0]); Az = np.array([-4.0])
    Bx = np.array([0.0]); By = np.array([0.0]); Bz = np.array([-5.0])
    X, Y, Z = rotate_A_so_z_aligns_with_B(Ax, Ay, Az, Bx, By, Bz)
    assert_allclose([X[0], Y[0], Z[0]], [2.0, -3.0, 4.0], atol=1e-12)

def test_zero_B_leaves_A_unchanged():
    Ax = np.array([1.,2.]); Ay = np.array([3.,4.]); Az = np.array([5.,6.])
    Bx = np.array([0.,0.]); By = np.array([0.,0.]); Bz = np.array([0.,0.])
    X, Y, Z = rotate_A_so_z_aligns_with_B(Ax, Ay, Az, Bx, By, Bz)
    assert_allclose(X, Ax); assert_allclose(Y, Ay); assert_allclose(Z, Az)


In [ ]:
test_antiparallel_case()